[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 96. The Resilient Network Design (Robust Optimization)
## Tier 3: The Advanced Algorithm (Grasshopper Optimization Algorithm)

This notebook applies the Grasshopper Optimization Algorithm (GOA) to a facility-selection problem under disruption scenarios. A population of candidate network designs moves through a continuous search space; a threshold converts each candidate into a binary decision about which facilities to open. The algorithm returns the best candidate it encountered according to the fitness score defined below. GOA is a stochastic search method, so it does not guarantee a globally optimal solution.

### Learning objectives

By the end of the notebook, you should be able to:

- Represent facilities, demand zones, and disruption scenarios as data.
- Evaluate service coverage and transport cost under each scenario.
- Explain how continuous grasshopper positions represent binary facility decisions.
- Trace the GOA population update, best-so-far tracking, and convergence measurements.
- Interpret the returned candidate in terms of the model's assumptions and limitations.

### Problem inputs and outputs

The inputs are eight potential facilities, four demand zones, and three disruption scenarios with specified probabilities, facility failures, blocked routes, and demand multipliers. The main run uses a population of 15 grasshoppers and 30 iterations. The output is the best facility-selection candidate encountered, its fitness, scenario-level performance, and histories of best fitness and population diversity.

### Model assumptions and scope

- A facility serves a demand zone through a direct facility-to-zone connection. A blocked route removes that connection in the corresponding scenario.
- Service potential is calculated separately for each demand zone using the best available facility. Facility capacity is applied to each zone independently; capacity is not allocated jointly across zones. Coverage is therefore a service-potential proxy, not a capacity-constrained flow allocation.
- Transport cost uses the least-cost available facility for each demand zone and is not limited by facility capacity. A zone with no available connection contributes no transport cost in that scenario, while its demand still remains in the coverage denominator.
- Scenario probabilities weight expected cost. The service metric used by the fitness function is the minimum coverage across scenarios.
- The 80% coverage target appears as a penalty in the scalar fitness function. It is not a hard feasibility constraint: the search can return a candidate below 80% if its score is better under that scalarization.
- The model does not include facility opening decisions that vary by scenario, inventory, production, explicit demand allocation, or unmet-demand penalties beyond the coverage term.

### GOA search outline

1. Initialize a population of continuous positions in the unit interval.
2. Convert each position to a facility-selection vector using a threshold of 0.5 and evaluate its fitness.
3. Calculate social interactions between grasshoppers and attraction toward the best position found so far.
4. Decrease the coefficient controlling movement over the iterations, update positions, and keep them within the unit interval.
5. Evaluate the new candidates and record best-so-far fitness and population diversity.

The social-interaction function combines attraction and repulsion as a function of distance. In this implementation, the movement update is a continuous GOA-inspired update for bounded decision coordinates; thresholding is used to evaluate the resulting facility selections. The social force is calculated from the population in its current order, so positions updated earlier in an iteration can affect later updates in that same iteration.

### Fitness model

Let $S$ be the set of selected facilities and let $k$ index disruption scenarios. The scenario-specific coverage is denoted by $q_k(S)$, the scenario-specific transport cost by $T_k(S)$, the fixed opening cost by $F(S)$, and the scenario probability by $p_k$. The expected cost and worst-scenario coverage are

$$
C(S) = \sum_k p_k\bigl(F(S) + T_k(S)\bigr),
\qquad
q_{\min}(S) = \min_k q_k(S).
$$

The implemented score is

$$
\operatorname{fitness}(S)
= C(S) + 1{,}000{,}000\max\bigl(0, 0.8-q_{\min}(S)\bigr)
  - 100{,}000q_{\min}(S).
$$

The algorithm minimizes this score. The first term rewards lower expected cost, the second penalizes coverage below 80%, and the third rewards higher worst-scenario coverage. These terms form a scalarized objective; they do not impose a strict 80% coverage constraint. Candidates selecting fewer than two facilities receive infinite fitness. In the code, these quantities are computed in `evaluate_fitness` using `total_expected_cost`, `worst_service`, `penalty`, and `fitness`.

### Interpreting the search histories

The convergence history records the best fitness found after each iteration, not the fitness of every grasshopper. The diversity history is the mean pairwise Euclidean distance between continuous position vectors. Lower diversity indicates that the population's positions have become more similar; it does not by itself prove that the population has found a high-quality or globally optimal design. Random initialization is used without a fixed seed, so repeated runs can produce different results.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """A potential facility location and its modeled attributes."""

    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float

    def distance_to(self, other) -> float:
        """Return Euclidean distance to an object with x and y coordinates."""
        if not hasattr(other, "x") or not hasattr(other, "y"):
            raise TypeError("The target must have x and y coordinates.")
        return float(np.hypot(self.x - other.x, self.y - other.y))


@dataclass
class DemandZone:
    """A demand location with a baseline demand quantity."""

    id: int
    x: float
    y: float
    demand: float


@dataclass
class Scenario:
    """A disruption scenario and its probability-weighted inputs."""

    id: int
    probability: float
    facility_failures: List[int]
    route_blocks: Set[Tuple[int, int]]
    demand_multipliers: Dict[int, float]


@dataclass
class Grasshopper:
    """A candidate represented by continuous facility-selection coordinates."""

    position: np.ndarray
    fitness: float
    velocity: np.ndarray

    def __init__(self, n_facilities: int):
        self.position = np.random.random(n_facilities)
        self.fitness = float("inf")
        # Velocity is retained as a state field; this position-update rule does not use it.
        self.velocity = np.zeros(n_facilities)


@dataclass
class GOAResult:
    """Best candidate and diagnostic histories returned by the optimizer."""

    best_position: np.ndarray
    best_fitness: float
    best_facilities: List[Facility]
    convergence_history: List[float]
    diversity_history: List[float]
    execution_time: float

In [ ]:
def create_goa_problem():
    """Create the facility-location instance used in this notebook."""
    facilities = [
        Facility(1, 10, 20, 1000, 200000),
        Facility(2, 30, 15, 1200, 250000),
        Facility(3, 25, 35, 1100, 230000),
        Facility(4, 45, 30, 1300, 280000),
        Facility(5, 15, 50, 900, 180000),
        Facility(6, 40, 10, 1400, 300000),
        Facility(7, 35, 45, 1150, 240000),
        Facility(8, 20, 40, 1050, 210000),
    ]

    demand_zones = [
        DemandZone(1, 15, 25, 200),
        DemandZone(2, 35, 20, 300),
        DemandZone(3, 25, 40, 250),
        DemandZone(4, 40, 35, 350),
    ]

    scenarios = [
        Scenario(
            id=1,
            probability=0.6,
            facility_failures=[],
            route_blocks=set(),
            demand_multipliers={1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0},
        ),
        Scenario(
            id=2,
            probability=0.25,
            facility_failures=[2, 6],
            route_blocks={(1, 3), (4, 2)},
            demand_multipliers={1: 1.5, 2: 0.7, 3: 1.3, 4: 0.8},
        ),
        Scenario(
            id=3,
            probability=0.15,
            facility_failures=[5],
            route_blocks={(3, 1), (7, 4)},
            demand_multipliers={1: 0.8, 2: 1.2, 3: 0.9, 4: 1.1},
        ),
    ]

    return facilities, demand_zones, scenarios


facilities, demand_zones, scenarios = create_goa_problem()

assert np.isclose(sum(scenario.probability for scenario in scenarios), 1.0)
assert all(scenario.probability >= 0 for scenario in scenarios)

print("GOA Problem Setup:")
print(f"- Facilities: {len(facilities)} potential locations")
print(f"- Demand zones: {len(demand_zones)} service areas")
print(f"- Scenarios: {len(scenarios)} disruption scenarios")
print("- Coverage target used in the fitness penalty: 80%")

In [ ]:
class GrasshopperNetworkOptimizer:
    """Grasshopper-based search for resilient facility-selection candidates."""

    def __init__(
        self,
        facilities: List[Facility],
        demand_zones: List[DemandZone],
        scenarios: List[Scenario],
        pop_size: int = 15,
        max_iter: int = 30,
    ):
        if not facilities:
            raise ValueError("At least one facility is required.")
        if pop_size < 1 or max_iter < 1:
            raise ValueError("Population size and iteration count must be positive.")

        self.facilities = facilities
        self.demand_zones = demand_zones
        self.scenarios = scenarios
        self.n_facilities = len(facilities)
        self.pop_size = pop_size
        self.max_iter = max_iter

        # These parameters control social-force scaling and interaction shape.
        self.c_max = 1.0
        self.c_min = 0.00004
        self.f = 0.5
        self.l = 1.5

        self.population = [Grasshopper(self.n_facilities) for _ in range(self.pop_size)]
        self.best_position = None
        self.best_fitness = float("inf")
        self.convergence_history = []
        self.diversity_history = []

    def social_interaction(self, distance: float) -> float:
        """Return the attraction-repulsion interaction strength at a distance."""
        if distance == 0:
            return 0.0
        return self.f * np.exp(-distance / self.l) - np.exp(-distance)

    def calculate_service_coverage(
        self,
        selected_facilities: List[Facility],
        scenario: Scenario,
    ) -> float:
        """Calculate the modeled demand-weighted service coverage for a scenario."""
        available_facilities = [
            facility
            for facility in selected_facilities
            if facility.id not in scenario.facility_failures
        ]

        total_demand = 0.0
        served_demand = 0.0

        for demand in self.demand_zones:
            actual_demand = demand.demand * scenario.demand_multipliers.get(
                demand.id, 1.0
            )
            total_demand += actual_demand
            best_service = 0.0

            for facility in available_facilities:
                if (facility.id, demand.id) in scenario.route_blocks:
                    continue

                distance = facility.distance_to(demand)
                service_factor = 1.0 / (1.0 + distance * 0.01)
                service = min(facility.capacity, actual_demand) * service_factor
                best_service = max(best_service, service)

            served_demand += best_service

        return served_demand / total_demand if total_demand > 0 else 0.0

    def calculate_transport_cost(
        self,
        selected_facilities: List[Facility],
        scenario: Scenario,
    ) -> float:
        """Sum each zone's least-cost available connection in a scenario."""
        available_facilities = [
            facility
            for facility in selected_facilities
            if facility.id not in scenario.facility_failures
        ]

        total_cost = 0.0
        for demand in self.demand_zones:
            actual_demand = demand.demand * scenario.demand_multipliers.get(
                demand.id, 1.0
            )
            minimum_cost = float("inf")

            for facility in available_facilities:
                if (facility.id, demand.id) in scenario.route_blocks:
                    continue

                distance = facility.distance_to(demand)
                cost = distance * 0.1 * actual_demand
                minimum_cost = min(minimum_cost, cost)

            if np.isfinite(minimum_cost):
                total_cost += minimum_cost

        return total_cost

    def evaluate_fitness(self, position: np.ndarray) -> float:
        """Threshold a position and evaluate its scalarized robust-network score."""
        threshold = 0.5
        selected_indices = np.where(position > threshold)[0]

        if len(selected_indices) < 2:
            return float("inf")

        selected_facilities = [self.facilities[index] for index in selected_indices]
        fixed_cost = sum(facility.fixed_cost for facility in selected_facilities)
        total_expected_cost = 0.0
        worst_service = 1.0

        for scenario in self.scenarios:
            service_coverage = self.calculate_service_coverage(
                selected_facilities, scenario
            )
            transport_cost = self.calculate_transport_cost(
                selected_facilities, scenario
            )
            scenario_cost = fixed_cost + transport_cost
            total_expected_cost += scenario.probability * scenario_cost
            worst_service = min(worst_service, service_coverage)

        penalty = 0.0
        if worst_service < 0.8:
            penalty = 1_000_000 * (0.8 - worst_service)

        fitness = total_expected_cost + penalty - (worst_service * 100_000)
        return float(fitness)

    def calculate_diversity(self) -> float:
        """Return mean pairwise Euclidean distance between population positions."""
        if len(self.population) < 2:
            return 0.0

        total_distance = 0.0
        pair_count = 0
        for first_index in range(len(self.population)):
            for second_index in range(first_index + 1, len(self.population)):
                distance = np.linalg.norm(
                    self.population[first_index].position
                    - self.population[second_index].position
                )
                total_distance += distance
                pair_count += 1

        return total_distance / pair_count if pair_count else 0.0

    def update_position(self, iteration: int):
        """Apply social movement, best-position attraction, and coordinate bounds."""
        coefficient = self.c_max - iteration * (
            (self.c_max - self.c_min) / self.max_iter
        )

        for index, grasshopper in enumerate(self.population):
            social_force = np.zeros(self.n_facilities)

            for other_index, other in enumerate(self.population):
                if index == other_index:
                    continue

                difference = other.position - grasshopper.position
                distance = np.linalg.norm(difference)
                if distance > 0:
                    unit_vector = difference / distance
                    social_strength = self.social_interaction(distance)
                    social_force += coefficient * (
                        (self.n_facilities / 2) * social_strength * unit_vector
                    )

            if self.best_position is not None:
                target_force = coefficient * (
                    self.best_position - grasshopper.position
                )
                social_force += target_force

            new_position = grasshopper.position + social_force
            grasshopper.position = np.clip(new_position, 0.0, 1.0)

            # The binary evaluation requires at least two selected locations.
            while np.sum(grasshopper.position > 0.5) < 2:
                inactive_indices = np.where(grasshopper.position <= 0.5)[0]
                if len(inactive_indices) == 0:
                    break
                activate_index = np.random.choice(inactive_indices)
                grasshopper.position[activate_index] = 0.7

    def optimize(self) -> GOAResult:
        """Run GOA and return the best candidate found during the search."""
        start_time = time.time()

        print("=" * 60)
        print("GRASSHOPPER OPTIMIZATION ALGORITHM")
        print("=" * 60)
        print(f"Population size: {self.pop_size}")
        print(f"Maximum iterations: {self.max_iter}")
        print(f"Problem size: {self.n_facilities} facilities")

        for grasshopper in self.population:
            grasshopper.fitness = self.evaluate_fitness(grasshopper.position)
            if grasshopper.fitness < self.best_fitness:
                self.best_fitness = grasshopper.fitness
                self.best_position = grasshopper.position.copy()

        print(f"\nInitial best fitness: {self.best_fitness:.2f}")

        for iteration in range(self.max_iter):
            self.update_position(iteration)

            for grasshopper in self.population:
                grasshopper.fitness = self.evaluate_fitness(grasshopper.position)
                if grasshopper.fitness < self.best_fitness:
                    self.best_fitness = grasshopper.fitness
                    self.best_position = grasshopper.position.copy()

            self.convergence_history.append(self.best_fitness)
            self.diversity_history.append(self.calculate_diversity())

            if iteration % 5 == 0:
                print(
                    f"Iteration {iteration + 1:2d}: "
                    f"Best fitness = {self.best_fitness:.2f}, "
                    f"Diversity = {self.diversity_history[-1]:.3f}"
                )

        execution_time = time.time() - start_time
        selected_indices = np.where(self.best_position > 0.5)[0]
        best_facilities = [self.facilities[index] for index in selected_indices]

        return GOAResult(
            best_position=self.best_position.copy(),
            best_fitness=self.best_fitness,
            best_facilities=best_facilities,
            convergence_history=self.convergence_history.copy(),
            diversity_history=self.diversity_history.copy(),
            execution_time=execution_time,
        )

In [ ]:
def run_goa(
    facilities: List[Facility],
    demand_zones: List[DemandZone],
    scenarios: List[Scenario],
):
    """Run the specified GOA instance and report its modeled performance."""
    goa = GrasshopperNetworkOptimizer(
        facilities,
        demand_zones,
        scenarios,
        pop_size=15,
        max_iter=30,
    )
    result = goa.optimize()

    print("\n" + "=" * 60)
    print("GOA OPTIMIZATION RESULTS")
    print("=" * 60)
    print("\nSelected Facilities:")
    for facility in result.best_facilities:
        print(
            f"  Facility {facility.id}: ({facility.x}, {facility.y}) - "
            f"Fixed cost: ${facility.fixed_cost:,.0f}"
        )

    fixed_cost = sum(facility.fixed_cost for facility in result.best_facilities)
    print("\nPerformance Analysis:")
    print(f"  Fixed Cost: ${fixed_cost:,.0f}")
    print(f"  Best Fitness: {result.best_fitness:.2f}")
    print(f"  Execution Time: {result.execution_time:.2f} seconds")

    worst_service = 1.0
    total_expected_cost = 0.0
    print("\nScenario Performance:")
    for scenario in scenarios:
        service_coverage = goa.calculate_service_coverage(
            result.best_facilities, scenario
        )
        transport_cost = goa.calculate_transport_cost(
            result.best_facilities, scenario
        )
        scenario_cost = fixed_cost + transport_cost
        total_expected_cost += scenario.probability * scenario_cost
        worst_service = min(worst_service, service_coverage)

        print(
            f"  Scenario {scenario.id}: "
            f"Coverage = {service_coverage:.1%}, "
            f"Cost = ${scenario_cost:,.0f}"
        )

    print("\nSummary:")
    print(f"  Expected Total Cost: ${total_expected_cost:,.0f}")
    print(f"  Worst-Case Coverage: {worst_service:.1%}")
    print(f"  80% Coverage Target: {'Met' if worst_service >= 0.8 else 'Not met'}")
    print("  The coverage target is represented by a fitness penalty, not a hard constraint.")

    return goa, result


goa, goa_result = run_goa(facilities, demand_zones, scenarios)

In [ ]:
def create_goa_visualization(
    facilities: List[Facility],
    demand_zones: List[DemandZone],
    goa: GrasshopperNetworkOptimizer,
    result: GOAResult,
):
    """Plot search histories, the returned network, and final positions."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Grasshopper Optimization for Resilient Network Design",
        fontsize=16,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    ax1.set_title("Best-So-Far Fitness", fontweight="bold")
    iterations = np.arange(1, len(result.convergence_history) + 1)
    ax1.plot(
        iterations,
        result.convergence_history,
        color="blue",
        linewidth=2,
        marker="o",
        markersize=4,
    )
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Fitness (lower is better)")
    ax1.grid(True, alpha=0.3)
    if len(result.convergence_history) > 1:
        initial_fitness = result.convergence_history[0]
        final_fitness = result.convergence_history[-1]
        if not np.isclose(initial_fitness, 0.0):
            improvement = (
                (initial_fitness - final_fitness) / abs(initial_fitness) * 100
            )
            ax1.annotate(
                f"Change from first recorded iteration: {improvement:.1f}%",
                xy=(0.5, 0.95),
                xycoords="axes fraction",
                ha="center",
                va="top",
                fontsize=9,
                bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8},
            )

    ax2 = axes[0, 1]
    ax2.set_title("Population Diversity", fontweight="bold")
    ax2.plot(
        iterations,
        result.diversity_history,
        color="green",
        linewidth=2,
        marker="s",
        markersize=4,
    )
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Mean Pairwise Position Distance")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    ax3.set_title("Returned Facility-Selection Candidate", fontweight="bold")
    selected_ids = {facility.id for facility in result.best_facilities}
    selected_facilities = [
        facility for facility in facilities if facility.id in selected_ids
    ]
    rejected_facilities = [
        facility for facility in facilities if facility.id not in selected_ids
    ]

    if selected_facilities:
        ax3.scatter(
            [facility.x for facility in selected_facilities],
            [facility.y for facility in selected_facilities],
            c="green",
            marker="o",
            s=180,
            edgecolors="black",
            linewidths=1,
            label="Selected facility",
        )
    if rejected_facilities:
        ax3.scatter(
            [facility.x for facility in rejected_facilities],
            [facility.y for facility in rejected_facilities],
            c="lightgray",
            marker="x",
            s=120,
            label="Not selected",
        )
    ax3.scatter(
        [demand.x for demand in demand_zones],
        [demand.y for demand in demand_zones],
        c="blue",
        marker="s",
        s=120,
        edgecolors="black",
        linewidths=1,
        label="Demand zone",
    )

    for facility in facilities:
        ax3.annotate(
            f"F{facility.id}",
            (facility.x, facility.y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )
    for demand in demand_zones:
        ax3.annotate(
            f"D{demand.id}",
            (demand.x, demand.y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )
    ax3.set_xlabel("X Coordinate")
    ax3.set_ylabel("Y Coordinate")
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc="best")
    ax3.set_aspect("equal", adjustable="datalim")

    ax4 = axes[1, 1]
    ax4.set_title("Final Population Position Matrix", fontweight="bold")
    position_matrix = np.array(
        [grasshopper.position for grasshopper in goa.population]
    )
    image = ax4.imshow(
        position_matrix,
        cmap="RdYlBu_r",
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )
    ax4.set_xlabel("Facility")
    ax4.set_ylabel("Grasshopper")
    ax4.set_xticks(range(len(facilities)))
    ax4.set_xticklabels([f"F{facility.id}" for facility in facilities])
    colorbar = fig.colorbar(image, ax=ax4)
    colorbar.set_label("Continuous Position Value (threshold = 0.5)")

    fig.tight_layout()
    plt.show()
    return fig


fig = create_goa_visualization(facilities, demand_zones, goa, goa_result)

In [ ]:
def compare_algorithms(
    goa: GrasshopperNetworkOptimizer,
    result: GOAResult,
    scenarios: List[Scenario],
):
    """Summarize this run and state qualitative method characteristics."""
    print("\n" + "=" * 60)
    print("ALGORITHM AND RUN SUMMARY")
    print("=" * 60)
    print("\nGrasshopper Optimization Algorithm:")
    print(f"  Selected Facilities: {[f.id for f in result.best_facilities]}")
    print(f"  Best Fitness: {result.best_fitness:.2f}")
    print(f"  Execution Time: {result.execution_time:.2f} seconds")
    print(f"  Recorded Iterations: {len(result.convergence_history)}")

    fixed_cost = sum(facility.fixed_cost for facility in result.best_facilities)
    worst_service = 1.0
    total_expected_cost = 0.0
    for scenario in scenarios:
        coverage = goa.calculate_service_coverage(result.best_facilities, scenario)
        transport_cost = goa.calculate_transport_cost(
            result.best_facilities, scenario
        )
        total_expected_cost += scenario.probability * (fixed_cost + transport_cost)
        worst_service = min(worst_service, coverage)

    print(f"  Expected Total Cost: ${total_expected_cost:,.0f}")
    print(f"  Worst-Case Coverage: {worst_service:.1%}")
    print("\nMethod Characteristics:")
    print("  - Population-based search evaluates multiple candidates per iteration.")
    print("  - The result is the best candidate encountered, not a proof of optimality.")
    print("  - Search behavior depends on the random initialization and GOA parameters.")
    print("  - The fitness score combines expected cost with a coverage penalty and reward.")
    print("  - This notebook does not contain execution results for other tiers, so no numerical cross-method comparison is made.")

    if result.convergence_history:
        print("\nConvergence Summary:")
        first_fitness = result.convergence_history[0]
        last_fitness = result.convergence_history[-1]
        print(f"  First recorded best fitness: {first_fitness:.2f}")
        print(f"  Final best fitness: {last_fitness:.2f}")
        print(f"  Change: {last_fitness - first_fitness:.2f}")

    if result.diversity_history:
        print("\nDiversity Summary:")
        print(f"  First recorded diversity: {result.diversity_history[0]:.3f}")
        print(f"  Final diversity: {result.diversity_history[-1]:.3f}")

    return result


comparison_result = compare_algorithms(goa, goa_result, scenarios)

In [ ]:
def parameter_sensitivity_analysis(
    facilities: List[Facility],
    demand_zones: List[DemandZone],
    scenarios: List[Scenario],
):
    """Compare the specified population sizes using the same GOA formulation."""
    print("\n" + "=" * 60)
    print("POPULATION-SIZE SENSITIVITY ANALYSIS")
    print("=" * 60)

    pop_sizes = [5, 10, 15, 20]
    results = []

    for pop_size in pop_sizes:
        goa_test = GrasshopperNetworkOptimizer(
            facilities,
            demand_zones,
            scenarios,
            pop_size=pop_size,
            max_iter=20,
        )
        result = goa_test.optimize()

        fixed_cost = sum(facility.fixed_cost for facility in result.best_facilities)
        worst_service = 1.0
        total_expected_cost = 0.0
        for scenario in scenarios:
            coverage = goa_test.calculate_service_coverage(
                result.best_facilities, scenario
            )
            transport_cost = goa_test.calculate_transport_cost(
                result.best_facilities, scenario
            )
            total_expected_cost += scenario.probability * (fixed_cost + transport_cost)
            worst_service = min(worst_service, coverage)

        results.append(
            {
                "pop_size": pop_size,
                "fitness": result.best_fitness,
                "cost": total_expected_cost,
                "coverage": worst_service,
                "time": result.execution_time,
                "facilities": len(result.best_facilities),
            }
        )
        print(
            f"Population {pop_size:2d}: Fitness={result.best_fitness:10.2f}, "
            f"Expected cost=${total_expected_cost:10.0f}, "
            f"Worst-case coverage={worst_service:6.1%}, "
            f"Time={result.execution_time:6.2f} s"
        )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("GOA Population-Size Sensitivity", fontsize=16, fontweight="bold")
    population_values = [result["pop_size"] for result in results]

    axes[0, 0].plot(
        population_values,
        [result["fitness"] for result in results],
        "bo-",
        linewidth=2,
        markersize=8,
    )
    axes[0, 0].set_xlabel("Population Size (grasshoppers)")
    axes[0, 0].set_ylabel("Best Fitness")
    axes[0, 0].set_title("Fitness by Population Size")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        population_values,
        [result["cost"] for result in results],
        "go-",
        linewidth=2,
        markersize=8,
    )
    axes[0, 1].set_xlabel("Population Size (grasshoppers)")
    axes[0, 1].set_ylabel("Expected Total Cost ($)")
    axes[0, 1].set_title("Expected Cost by Population Size")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        population_values,
        [result["coverage"] * 100 for result in results],
        "mo-",
        linewidth=2,
        markersize=8,
    )
    axes[1, 0].axhline(
        y=80,
        color="red",
        linestyle="--",
        alpha=0.7,
        label="80% target",
    )
    axes[1, 0].set_xlabel("Population Size (grasshoppers)")
    axes[1, 0].set_ylabel("Worst-Case Coverage (%)")
    axes[1, 0].set_title("Coverage by Population Size")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(
        population_values,
        [result["time"] for result in results],
        "co-",
        linewidth=2,
        markersize=8,
    )
    axes[1, 1].set_xlabel("Population Size (grasshoppers)")
    axes[1, 1].set_ylabel("Execution Time (seconds)")
    axes[1, 1].set_title("Runtime by Population Size")
    axes[1, 1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return results


sensitivity_results = parameter_sensitivity_analysis(
    facilities, demand_zones, scenarios
)

## Position of this tier in the solution sequence

This tier uses a population-based metaheuristic to search facility-selection candidates. Its search is guided by interactions among grasshoppers and attraction toward the best candidate found so far. This can be useful when the candidate space is difficult to explore with a simple local or rule-based procedure, but the method does not certify optimality. The performance depends on the population, iteration limit, interaction parameters, and random initialization.

The notebook's sensitivity experiment varies population size while retaining 20 iterations for each run. It illustrates how the observed score, coverage, cost, and runtime can depend on a search setting; it is not a guarantee that a larger population will produce a better solution. Since no results from other tiers are computed here, any cross-tier comparison must be qualitative rather than numerical.

## Strengths and limitations

**Potential strengths**

- Multiple candidates are explored within one run.
- Social interaction and attraction to the best-so-far position provide a structured search direction.
- The continuous position representation can be thresholded into binary facility decisions.
- Fitness and diversity histories provide complementary views of search progress.

**Important limitations**

- There is no guarantee that the returned candidate is globally optimal.
- The fitness function is a scalarized score, so its penalty and reward coefficients influence the trade-off between cost and coverage.
- Meeting the 80% target is not guaranteed because coverage is penalized rather than enforced as a hard constraint.
- The modeled coverage and transport cost do not jointly allocate facility capacity across demand zones.
- A single stochastic run is not evidence of repeatable performance; this notebook does not fix a random seed or aggregate repeated runs.

## When this approach may be useful

A population-based search can be considered for facility-location settings with complicated objective landscapes or many interacting design choices, particularly when a practitioner can evaluate candidate designs but does not require an optimality certificate. Before applying the model operationally, the service, capacity, route, and disruption assumptions should be checked against the network being represented.